In [377]:
import requests
from langchain_core.prompts import PromptTemplate 
from langchain_core.tools import tool, StructuredTool, InjectedToolArg
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from langchain.agents import create_agent
from pydantic import BaseModel, Field
from typing import Annotated
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
import os
from dotenv import load_dotenv
import yfinance as yf
import pandas_ta as ta
import json

load_dotenv()

True

In [326]:
NEWS_DATA_API=os.getenv('NEWS_DATA_API')

In [328]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
    template="""You are a query parameter builder for the NewsData.io API focusing strictly on Indian stock market catalysts (NSE/BSE).

Target Topic: {topic}

Step 1: Choose your search scope (CHOOSE EXACTLY ONE):
- 'q=...' (Broad search across the whole article)
- 'qInTitle=...' (Strict search only in the headline)
- 'qInMeta=...' (Search titles, URLs, and meta tags)
- 'timeframe=...' (Search by timeframe)
- 'category=...' (Search by category)

CRITICAL: Never combine these. Pick only one.

Step 2: Apply these mandatory rules:
1. Use actionable single-stock catalyst terms (e.g., "quarterly results", "order win", "acquisition").
2. Strictly DO NOT include broad market phrases like "Sensex" or "Nifty".
3. Always append '&category=business&country=in'.

Validation Rules:
- STRICTLY EXCLUDE foreign equities (e.g., Nike, Tesla, Apple, Google). Only include companies actively listed on the NSE or BSE.
- STRICTLY EXCLUDE general editorial comparisons or educational articles without fresh catalysts.
- For each valid event, state the exact NSE/BSE ticker symbol (e.g., UNIONBANK, INFY, WIPRO).

Output Format:
- Output ONLY the final query string separated by '&'.
- Do NOT include the base URL or conversational explanations.
- Example output: qInTitle=order+win+OR+acquisition&category=business&country=in
""",
    input_variables=["topic"]
)

In [329]:
model = ChatOllama(model="gemma4:e4b", temperature=0.7, num_ctx=8192)

In [330]:
parser = StrOutputParser()

In [331]:
class Article(BaseModel):
    title: str = Field(..., description="The title of the news article"),
    description: str = Field(..., description="A brief description of the news article"),
    source: str = Field(..., description="The source of the news article")

In [332]:
class ArticleList(BaseModel):
    articles: list[Article] = Field(description="List of 5 news articles")

In [333]:
articles_parser = PydanticOutputParser(pydantic_object=ArticleList)


In [334]:
@tool
def generate_query_params(topic: str):
    """Generate a formatted URL query parameter string for fetching news articles."""
    # print("generate_query_params called with topic:", topic)
    model = ChatOllama(model="gemma4:e4b", temperature=0.7, max_tokens=4000)
    chain = prompt | model | parser

    prompt_text = prompt.invoke({'topic': topic})
    print(prompt_text)

    query_params = chain.invoke({'topic' : topic})
    # print(query_params)

    return query_params

In [335]:
class NewsQueryParams(BaseModel):
    topic : str = Field(..., description="The topic to search news articles for")

# @tool
def fetch_news_articles(topic: str):
    """ Tool to fetch news articles. MUST only be called after generating query parameters. """

    query_chain = prompt | model | parser
    query_params = query_chain.invoke({"topic": topic})

    print("query_params_before_fetching: ",query_params)
    
    url = f"https://newsdata.io/api/1/latest?apikey={NEWS_DATA_API}&{query_params}&size=10"
    response = requests.get(url)
    data = response.json()
    
    # 1. Check for API success
    if data.get("status") == "success":
        clean_articles = []
        
        # 2. Extract only the essential text needed by the financial analyst prompt
        for article in data.get("results", []):
            clean_articles.append({
                "title": article.get("title"),
                "description": article.get("description"),
                "source": article.get("source_name")
            })
            
        # 3. Return the minimized list of dictionaries, NOT the raw JSON
        return clean_articles

    return data

fetch_news_articles_tool = StructuredTool.from_function(
    func=fetch_news_articles,
    name="fetch_news_articles",
    description="Fetch relevant news articles by passing relevant query parameters to the API",
    args_schema=NewsQueryParams
)

In [336]:
agent = create_agent(
    model=model,
    tools=[fetch_news_articles_tool],
)

format_instructions = articles_parser.get_format_instructions()

prompt_text = f"""You are an equity research analyst covering individual stocks on the NSE and BSE.

Step 1:
Call `fetch_news_articles` with a query focused on company-specific corporate actions (such as quarterly earnings surprises, mega contract wins, mergers, or regulatory penalties).

Step 2:
From the fetched results, filter out and ignore:
- Broad market index wrap-ups (e.g., Sensex/Nifty up/down commentary).
- Trading holiday calendars and exchange scheduling notices.
- Global macroeconomic recaps (crude oil prices, US bond yields).

Step 3:
Identify 5 specific companies with actionable catalysts capable of moving the individual stock price by 2% or more.

Output strictly according to these formatting instructions:
{format_instructions}"""

response = agent.invoke({"messages": [HumanMessage(content=prompt_text)]})
raw_content = response["messages"][-1].content
# print("Raw content from agent:\n", raw_content)
structured_result = articles_parser.parse(raw_content)

for i, item in enumerate(structured_result.articles):
    print(f"Article {i+1}:\nTitle: {item.title}\nDescription: {item.description}\nSource: {item.source}\n")

c:\YASHIT\AI\market-announcements\.venv\Lib\site-packages\pydantic\json_schema.py:2463: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=True, description='The title of the news article'),) is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
c:\YASHIT\AI\market-announcements\.venv\Lib\site-packages\pydantic\json_schema.py:2463: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=True, description='A brief description of the news article'),) is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)


query_params_before_fetching:  qInTitle=quarterly+results+OR+earnings+surprise+OR+order+win+OR+acquisition+OR+merger+OR+regulatory+penalty&category=business&country=in
Article 1:
Title: Infosys Q4 Earnings Beat Expectations on Strong Digital Transformation Demand
Description: Infosys reported a 15% year-on-year revenue jump, significantly surpassing analyst estimates due to robust demand in the AI and cloud segments, presenting a strong upward catalyst for the stock.
Source: Business Standard

Article 2:
Title: Larsen & Toubro Secures Mega Contract for Coastal Defense Project
Description: L&T has won a substantial, multi-year contract from the Ministry of Defence for a major coastal defense system, expected to boost order book visibility and revenue by over 20%.
Source: Economic Times

Article 3:
Title: Adani Ports Announces Strategic Acquisition of Minority Stake in Major Shipping Line
Description: The acquisition move strengthens Adani Ports' logistics footprint in the regional marke

In [356]:
class tickers_technical_scores(BaseModel):
    ticker: str = Field(..., description="The stock ticker symbol")
    open: float = Field(..., description="The opening price of the stock")
    close: float = Field(..., description="The closing price of the stock")
    high: float = Field(..., description="The highest price of the stock during the trading session")
    low: float = Field(..., description="The lowest price of the stock during the trading session")
    rsi14: float = Field(..., description="The 14-day Relative Strength Index (RSI) value for the stock")
    macd12_26_9: float = Field(..., description="The MACD value calculated using the 12-day and 26-day exponential moving averages with a 9-day signal line")
    ema20: float = Field(..., description="The 20-day Exponential Moving Average value for the stock")
    news: Article = Field(..., description="Title + Description for lates update on ticker")
    conclusion : str = Field(..., description="final conclusion on the basis of news + techincal score for the next day")

In [357]:

@tool
def get_ticker_technicals(ticker: str):
    """ This tool return the technicals scores like RSI, MACD, EMA """
    print("ticker: " , ticker)
    ticker_clean = ticker.strip().upper()
    if not ticker_clean.endswith(".NS"):
        ticker_clean = f"{ticker_clean}.NS"

    df = yf.Ticker(ticker_clean).history(period="6mo")

    # Single-line built-in calls that attach new columns to df:
    df.ta.rsi(length=14, append=True)
    df.ta.macd(fast=12, slow=26, signal=9, append=True)
    df.ta.ema(length=20, append=True)

    # print("data frame header: ", df.info())

    # Access generated values
    latest_rsi = df["RSI_14"].iloc[-1]
    latest_macd = df["MACD_12_26_9"].iloc[-1]
    latest_signal = df["MACDs_12_26_9"].iloc[-1]
    latest_hist = df["MACDh_12_26_9"].iloc[-1]
    latest_ema20 = df["EMA_20"].iloc[-1]
    latest_open = df["Open"].iloc[-1]
    latest_close = df["Close"].iloc[-1]
    latest_high = df["High"].iloc[-1]
    latest_low = df["Low"].iloc[-1]

    return {
        'latest_rsi': latest_rsi,
        "latest_macd": latest_macd,
        "latest_signal": latest_signal,
        "latest_hist": latest_hist,
        "latest_ema20": latest_ema20,
        "latest_open": latest_open,
        "latest_close": latest_close,
        "latest_high": latest_high,
        "latest_low": latest_low
    }

In [378]:
agent_technical = create_agent(
    model=model,
    tools=[get_ticker_technicals]
)


ticker_technical_parser = PydanticOutputParser(pydantic_object=tickers_technical_scores)

articles_text = "\n".join([
    f"- {art.title} (Source: {art.source}): {art.description}"
    for art in structured_result.articles
])

prompt_content = f"""
You are a technical market analyst.
    Review the following news articles to identify the relevant NSE tickers:
    {articles_text}

    For each identified company:
    1. Call the `get_ticker_technicals` tool to fetch technical data.
    2. Evaluate the technicals against the news catalyst.

    Return your final answer strictly following these formatting instructions:
    {ticker_technical_parser.get_format_instructions()}
"""

agent_response = agent_technical.invoke({
    "messages": [HumanMessage(content=prompt_content)]
})

final_raw_text = agent_response["messages"][-1].content
clean_final_raw_data = final_raw_text.replace("```json", "").replace("```", "").strip()
parsed_data = json.loads(clean_final_raw_data)

# Inspect clean structured objects
for item in parsed_data:
    print(f"item: {item['ticker']} | item : {item['conclusion']} ")

c:\YASHIT\AI\market-announcements\.venv\Lib\site-packages\pydantic\json_schema.py:2463: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=True, description='The title of the news article'),) is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
c:\YASHIT\AI\market-announcements\.venv\Lib\site-packages\pydantic\json_schema.py:2463: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=True, description='A brief description of the news article'),) is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)


ticker:  INFY
ticker:  LT
ticker:  ADANIPORTS
ticker:  SUNPHARMA
ticker:  BAJFINANCE
item: INFY | item : The strong fundamental catalyst from the Q4 earnings beat and robust demand provides significant upward momentum. Despite current technical indicators being neutral/slightly weak, the positive news suggests a likely upward move for the next day. 
item: LT | item : The mega contract announcement is an extremely strong fundamental tailwind. This positive news, combined with the stock being in an oversold technical condition (RSI < 35), strongly suggests a bullish reversal and a positive move for the next day. 
item: ADANIPORTS | item : The positive strategic acquisition provides a solid fundamental catalyst. Despite the stock currently trading below the EMA20, the improved operational synergies suggest that the positive news will likely provide support and lead to a positive short-term reaction. 
item: SUNPHARMA | item : The long-term potential unlocked by the major pipeline fast-trac

In [376]:
import json
# print(final_raw_text)
clean_final_raw_data = final_raw_text.replace("```json", "").replace("```", "").strip()
parsed_data = json.loads(clean_final_raw_data)
# print(parsed_data)
for item in parsed_data:
    print(item)

{'ticker': 'INFY', 'open': 1010.0, 'close': 1035.0, 'high': 1035.0, 'low': 999.1500244140625, 'rsi14': 44.8781454431904, 'macd12_26_9': -26.398249819196735, 'ema20': 1043.684127361002, 'news': {'source': 'Business Standard', 'title': 'Infosys Q4 Earnings Beat Expectations on Strong Digital Transformation Demand', 'description': 'Infosys reported a 15% year-on-year revenue jump, significantly surpassing analyst estimates due to robust demand in the AI and cloud segments, presenting a strong upward catalyst for the stock.'}, 'conclusion': 'Bullish. The strong fundamental catalyst from Q4 earnings beat and robust AI/cloud demand is expected to override the current short-term bearish momentum indicated by the MACD.'}
{'ticker': 'LT', 'open': 3726.0, 'close': 3693.39990234375, 'high': 3737.300048828125, 'low': 3655.89990234375, 'rsi14': 31.276038787026952, 'macd12_26_9': -61.353333956232746, 'ema20': 3859.2746403201086, 'news': {'source': 'Economic Times', 'title': 'Larsen & Toubro Secures 